In [1]:

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from utils.config import (
    SEED,
    NUM_CLASSES,
    PRETRAINED_MAX_LEN,
    PRETRAINED_LR,
    NUM_WORKERS,
    RESULTS_DIR,
    device,
    set_seed,
)


# ============================================================
# 1. Configuration
# ============================================================

MODEL_NAME = "google-bert/bert-base-uncased"

MAX_LENGTH = PRETRAINED_MAX_LEN
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
EPOCHS = 3

LEARNING_RATE = PRETRAINED_LR
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

FOCAL_GAMMA = 2.0

TRAIN_PATH = Path("results/splits/train.csv")
VAL_PATH = Path("results/splits/validation.csv")

OUTPUT_DIR = (
    RESULTS_DIR / "optimization" / "loss_comparison"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = ["ce", "weighted_ce", "focal"]


# ============================================================
# 2. Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ============================================================
# 3. Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, df):
        self.texts = (
            df["text"].fillna("").astype(str).tolist()
        )
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "labels": self.labels[idx],
        }


class DynamicPaddingCollator:

    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, items):
        texts = [x["text"] for x in items]
        labels = [x["labels"] for x in items]

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        batch["labels"] = torch.tensor(
            labels, dtype=torch.long
        )
        return batch


def move_batch(batch):
    return {
        key: value.to(device)
        for key, value in batch.items()
    }


# ============================================================
# 4. Loss Functions
# ============================================================

class FocalLoss(nn.Module):
    """
    FL = alpha_y * (1 - p_t)^gamma * CE

    alpha_y: class weight for the true class
    p_t: predicted probability of the true class
    gamma: how strongly easy examples are down-weighted
    """

    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.register_buffer(
            "class_weights", class_weights.float()
        )
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets, reduction="none"
        )

        p_t = torch.exp(-ce).clamp(1e-8, 1.0)
        alpha_t = self.class_weights[targets]

        focal_factor = (1.0 - p_t).pow(self.gamma)

        numerator = (
            alpha_t * focal_factor * ce
        ).sum()

        denominator = alpha_t.sum().clamp_min(1e-8)

        return numerator / denominator


def make_criterion(loss_name, class_weights):
    if loss_name == "ce":
        return nn.CrossEntropyLoss()

    if loss_name == "weighted_ce":
        return nn.CrossEntropyLoss(
            weight=class_weights
        )

    if loss_name == "focal":
        return FocalLoss(
            class_weights=class_weights,
            gamma=FOCAL_GAMMA,
        )

    raise ValueError(f"Unknown loss: {loss_name}")


# ============================================================
# 5. Validation
# ============================================================

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_count = 0
    y_true, y_pred = [], []

    for batch in tqdm(
        loader, desc="Validation", leave=False
    ):
        batch = move_batch(batch)
        labels = batch["labels"]

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        count = labels.size(0)
        total_loss += loss.item() * count
        total_count += count

        predictions = logits.argmax(dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "val_loss": total_loss / max(total_count, 1),
        "val_accuracy": report["accuracy"],
        "val_macro_precision":
            report["macro avg"]["precision"],
        "val_macro_recall":
            report["macro avg"]["recall"],
        "val_macro_f1":
            report["macro avg"]["f1-score"],
        "val_weighted_f1":
            report["weighted avg"]["f1-score"],
    }

    return metrics, y_true, y_pred, report


# ============================================================
# 6. Train One Epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    epoch,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_count = 0
    num_batches = len(loader)

    use_amp = device.type == "cuda"

    for step, batch in enumerate(
        tqdm(loader, desc=f"Train epoch {epoch}")
    ):
        batch = move_batch(batch)
        labels = batch["labels"]

        # Correctly handle an incomplete final accumulation group.
        group_start = (
            step // GRAD_ACCUM_STEPS
        ) * GRAD_ACCUM_STEPS

        actual_group_size = min(
            GRAD_ACCUM_STEPS,
            num_batches - group_start,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            loss = criterion(outputs.logits, labels)

        loss_for_backward = loss / actual_group_size

        if use_amp:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        count = labels.size(0)
        total_loss += loss.detach().item() * count
        total_count += count

        should_update = (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or (step + 1) == num_batches
        )

        if should_update:
            if use_amp:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    return total_loss / max(total_count, 1)


# ============================================================
# 7. Save Validation Artifacts
# ============================================================

def save_confusion_matrix(y_true, y_pred, path, title):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )
    display.plot(values_format="d", xticks_rotation=45)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def save_curves(history, folder, loss_name):
    epochs = history["epoch"]

    plt.figure(figsize=(9, 5))
    plt.plot(
        epochs, history["train_loss"],
        marker="o", label="Train Loss"
    )
    plt.plot(
        epochs, history["val_loss"],
        marker="o", label="Validation Loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{loss_name}: Loss Curve")
    plt.xticks(epochs)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(folder / "training_curve.png", dpi=150)
    plt.close()

    plt.figure(figsize=(9, 5))
    plt.plot(
        epochs, history["val_macro_f1"],
        marker="o", label="Validation Macro F1"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{loss_name}: Validation Macro F1")
    plt.xticks(epochs)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(folder / "validation_macro_f1.png", dpi=150)
    plt.close()


# ============================================================
# 8. Run One Experiment
# ============================================================

def run_experiment(
    loss_name,
    train_df,
    val_df,
    tokenizer,
    class_weights,
):
    seed_everything(SEED)

    folder = OUTPUT_DIR / loss_name
    folder.mkdir(parents=True, exist_ok=True)

    checkpoint_path = folder / "best.pt"

    train_dataset = EmotionDataset(train_df)
    val_dataset = EmotionDataset(val_df)

    collator = DynamicPaddingCollator(
        tokenizer, MAX_LENGTH
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
    ).to(device)

    criterion = make_criterion(
        loss_name, class_weights.to(device)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    steps_per_epoch = (
        len(train_loader) + GRAD_ACCUM_STEPS - 1
    ) // GRAD_ACCUM_STEPS

    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda"),
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_precision": [],
        "val_macro_recall": [],
        "val_macro_f1": [],
        "val_weighted_f1": [],
        "epoch_seconds": [],
    }

    best_f1 = -1.0
    best_epoch = -1
    start_time = time.perf_counter()

    print("\n" + "=" * 70)
    print(f"Experiment: {loss_name}")
    print(f"Device: {device}")
    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(val_df):,}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Effective batch size: approximately "
          f"{BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()

        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scheduler,
            scaler,
            epoch,
        )

        val_metrics, y_true, y_pred, report = evaluate(
            model, val_loader, criterion
        )

        epoch_seconds = time.perf_counter() - epoch_start

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_accuracy"].append(
            val_metrics["val_accuracy"]
        )
        history["val_macro_precision"].append(
            val_metrics["val_macro_precision"]
        )
        history["val_macro_recall"].append(
            val_metrics["val_macro_recall"]
        )
        history["val_macro_f1"].append(
            val_metrics["val_macro_f1"]
        )
        history["val_weighted_f1"].append(
            val_metrics["val_weighted_f1"]
        )
        history["epoch_seconds"].append(epoch_seconds)

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_metrics['val_loss']:.4f}")
        print(f"Val Accuracy: {val_metrics['val_accuracy']:.4f}")
        print(f"Val Macro F1: {val_metrics['val_macro_f1']:.4f}")
        print(f"Epoch Time: {epoch_seconds / 60:.2f} min")

        if val_metrics["val_macro_f1"] > best_f1:
            best_f1 = val_metrics["val_macro_f1"]
            best_epoch = epoch

            torch.save(
                {
                    "epoch": best_epoch,
                    "model_state_dict": model.state_dict(),
                    "best_val_macro_f1": best_f1,
                    "loss_name": loss_name,
                    "model_name": MODEL_NAME,
                    "seed": SEED,
                    "max_length": MAX_LENGTH,
                    "num_classes": NUM_CLASSES,
                    "learning_rate": LEARNING_RATE,
                    "batch_size": BATCH_SIZE,
                    "grad_accum_steps": GRAD_ACCUM_STEPS,
                    "weight_decay": WEIGHT_DECAY,
                    "warmup_ratio": WARMUP_RATIO,
                    "focal_gamma": (
                        FOCAL_GAMMA if loss_name == "focal"
                        else None
                    ),
                    "class_weights": class_weights.tolist(),
                },
                checkpoint_path,
            )

            pd.DataFrame(report).T.to_csv(
                folder / "best_validation_report.csv",
                encoding="utf-8-sig",
            )

            save_confusion_matrix(
                y_true,
                y_pred,
                folder / "best_validation_confusion_matrix.png",
                f"{loss_name} Validation Confusion Matrix",
            )

            print(f"Saved best checkpoint: {checkpoint_path}")

    total_minutes = (
        time.perf_counter() - start_time
    ) / 60

    pd.DataFrame(history).to_csv(
        folder / "training_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    save_curves(history, folder, loss_name)

    result = {
        "loss_name": loss_name,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_f1,
        "training_minutes": total_minutes,
        "checkpoint_path": str(checkpoint_path),
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "focal_gamma": (
            FOCAL_GAMMA if loss_name == "focal"
            else None
        ),
    }

    with open(
        folder / "experiment_config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    del model, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 9. Main
# ============================================================

def main():
    set_seed(SEED)
    seed_everything(SEED)

    if not TRAIN_PATH.exists():
        raise FileNotFoundError(TRAIN_PATH)

    if not VAL_PATH.exists():
        raise FileNotFoundError(VAL_PATH)

    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)

    for name, df in [
        ("train", train_df),
        ("validation", val_df),
    ]:
        missing = {"text", "label"} - set(df.columns)
        if missing:
            raise ValueError(
                f"{name} split is missing columns: {missing}"
            )

    train_labels = train_df["label"].astype(int).to_numpy()

    expected_classes = np.arange(NUM_CLASSES)
    actual_classes = np.unique(train_labels)

    if not np.array_equal(actual_classes, expected_classes):
        raise ValueError(
            f"Expected labels {expected_classes.tolist()}, "
            f"found {actual_classes.tolist()}"
        )

    # Compute class weights from TRAIN labels only.
    weights_np = compute_class_weight(
        class_weight="balanced",
        classes=expected_classes,
        y=train_labels,
    )
    class_weights = torch.tensor(
        weights_np, dtype=torch.float32
    )

    print("Training class weights:")
    for class_id, weight in enumerate(weights_np):
        print(f"Class {class_id}: {weight:.4f}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    results = []

    for loss_name in EXPERIMENTS:
        result = run_experiment(
            loss_name=loss_name,
            train_df=train_df,
            val_df=val_df,
            tokenizer=tokenizer,
            class_weights=class_weights,
        )

        results.append(result)

        # Save after every experiment in case a later run is interrupted.
        pd.DataFrame(results).to_csv(
            OUTPUT_DIR / "loss_comparison_summary.csv",
            index=False,
            encoding="utf-8-sig",
        )

    results_df = pd.DataFrame(results).sort_values(
        "best_val_macro_f1",
        ascending=False,
    )

    results_df.to_csv(
        OUTPUT_DIR / "loss_comparison_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("\n" + "=" * 70)
    print("Loss comparison — sorted by Validation Macro F1")
    print("=" * 70)

    print(
        results_df[
            [
                "loss_name",
                "best_epoch",
                "best_val_macro_f1",
                "training_minutes",
            ]
        ].to_string(index=False)
    )

    print("\nTest data was not loaded or evaluated.")
    print("Use validation results to select the loss function.")


if __name__ == "__main__":
    main()

Training class weights:
Class 0: 0.5733
Class 1: 0.4929
Class 2: 2.0086
Class 3: 1.2125
Class 4: 1.4523
Class 5: 4.6306


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: ce
Device: cuda
Train rows: 305,345
Validation rows: 65,422
Learning rate: 2e-05
Effective batch size: approximately 64


Train epoch 1: 100%|██████████| 19085/19085 [07:41<00:00, 41.37it/s] 


...


Epoch 1/3
Train Loss: 0.1613
Val Loss: 0.0312
Val Accuracy: 0.9419
Val Macro F1: 0.9172
Epoch Time: 8.92 min
Saved best checkpoint: results/optimization/loss_comparison/focal/best.pt


Train epoch 2: 100%|██████████| 19085/19085 [07:20<00:00, 43.34it/s] 



Epoch 2/3
Train Loss: 0.0295
Val Loss: 0.0294
Val Accuracy: 0.9427
Val Macro F1: 0.9179
Epoch Time: 8.90 min
Saved best checkpoint: results/optimization/loss_comparison/focal/best.pt


Train epoch 3: 100%|██████████| 19085/19085 [07:23<00:00, 43.06it/s] 



Epoch 3/3
Train Loss: 0.0261
Val Loss: 0.0299
Val Accuracy: 0.9430
Val Macro F1: 0.9183
Epoch Time: 8.91 min
Saved best checkpoint: results/optimization/loss_comparison/focal/best.pt

Loss comparison — sorted by Validation Macro F1
  loss_name  best_epoch  best_val_macro_f1  training_minutes
      focal           3           0.918300         27.077965
weighted_ce           3           0.918248         26.688134
         ce           1           0.910447         28.289896

Test data was not loaded or evaluated.
Use validation results to select the loss function.


In [2]:
"""
Stage 07-2: Learning Rate Comparison
Model: BERT-base-uncased
Loss: Class-Weighted Focal Loss

Learning rates:
    1e-5
    2e-5
    3e-5

Uses:
    results/splits/train.csv
    results/splits/validation.csv

Test data is intentionally not loaded.
"""

import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "google-bert/bert-base-uncased"

NUM_CLASSES = 6
MAX_LENGTH = 128

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4

EPOCHS = 3

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

FOCAL_GAMMA = 2.0

LEARNING_RATES = [1e-5, 2e-5, 3e-5]

NUM_WORKERS = 0

TRAIN_PATH = Path("results/splits/train.csv")
VALIDATION_PATH = Path("results/splits/validation.csv")

OUTPUT_ROOT = Path(
    "results/optimization/learning_rate_comparison"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ============================================================
# 3. Load fixed splits
# ============================================================

def load_split(path, split_name):

    if not path.exists():
        raise FileNotFoundError(
            f"{split_name} file not found: {path.resolve()}"
        )

    df = pd.read_csv(path)

    required_columns = {"text", "label"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{path} is missing columns: {missing_columns}"
        )

    df = df[["text", "label"]].copy()

    df = df.dropna(subset=["label"])

    df["text"] = df["text"].fillna("").astype(str)
    df["label"] = df["label"].astype(int)

    invalid_labels = sorted(
        set(df["label"].unique()) - set(range(NUM_CLASSES))
    )

    if invalid_labels:
        raise ValueError(
            f"Invalid labels in {path}: {invalid_labels}"
        )

    if df.empty:
        raise ValueError(f"{split_name} dataset is empty.")

    return df.reset_index(drop=True)


# ============================================================
# 4. Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, dataframe):

        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        return {
            "text": self.texts[index],
            "labels": self.labels[index],
        }


# ============================================================
# 5. Dynamic padding collator
# ============================================================

class DynamicPaddingCollator:

    def __init__(self, tokenizer, max_length):

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):

        texts = [
            item["text"]
            for item in batch
        ]

        labels = torch.tensor(
            [item["labels"] for item in batch],
            dtype=torch.long,
        )

        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        encoded["labels"] = labels

        return encoded


# ============================================================
# 6. Class-Weighted Focal Loss
# ============================================================

class ClassWeightedFocalLoss(torch.nn.Module):

    def __init__(self, class_weights, gamma=2.0):

        super().__init__()

        self.register_buffer(
            "class_weights",
            class_weights,
        )

        self.gamma = gamma

    def forward(self, logits, targets):

        # Per-sample ordinary cross entropy
        ce = F.cross_entropy(
            logits,
            targets,
            reduction="none",
        )

        # Probability assigned to the correct class
        p_t = torch.exp(-ce).clamp(
            min=1e-8,
            max=1.0,
        )

        # Weight corresponding to each sample's true class
        alpha_t = self.class_weights[targets]

        # Focus more on difficult samples
        focal_factor = (1.0 - p_t).pow(self.gamma)

        numerator = (
            alpha_t * focal_factor * ce
        ).sum()

        denominator = alpha_t.sum().clamp_min(1e-8)

        return numerator / denominator


# ============================================================
# 7. Validation
# ============================================================

@torch.no_grad()
def evaluate(model, dataloader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    for batch in dataloader:

        labels = batch.pop("labels").to(DEVICE)

        inputs = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            outputs = model(**inputs)

            loss = criterion(
                outputs.logits,
                labels,
            )

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        predictions = outputs.logits.argmax(dim=-1)

        all_labels.extend(
            labels.cpu().tolist()
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)

    metrics = {
        "validation_loss": (
            total_loss / max(total_samples, 1)
        ),

        "accuracy": accuracy_score(
            y_true, y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "weighted_precision": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "weighted_recall": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    return metrics, y_true, y_pred


# ============================================================
# 8. Save best validation artifacts
# ============================================================

def save_validation_artifacts(
    output_dir,
    model,
    tokenizer,
    y_true,
    y_pred,
):

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose()

    report_df.to_csv(
        output_dir / "best_validation_report.csv"
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"true_{i}"
            for i in range(NUM_CLASSES)
        ],
        columns=[
            f"pred_{i}"
            for i in range(NUM_CLASSES)
        ],
    )

    cm_df.to_csv(
        output_dir / "best_validation_confusion_matrix.csv"
    )

    # Save tokenizer for future inference
    tokenizer.save_pretrained(
        output_dir / "tokenizer"
    )


# ============================================================
# 9. Run one learning-rate experiment
# ============================================================

def run_experiment(
    learning_rate,
    train_df,
    validation_df,
    tokenizer,
    class_weights_np,
):

    lr_tag = f"lr_{learning_rate:.0e}"

    output_dir = OUTPUT_ROOT / lr_tag
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Reset seed before each experiment.
    # This helps keep classifier initialization and data order comparable.
    set_seed(SEED)

    train_dataset = EmotionDataset(train_df)
    validation_dataset = EmotionDataset(validation_df)

    collator = DynamicPaddingCollator(
        tokenizer,
        MAX_LENGTH,
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        generator=generator,
        drop_last=False,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        drop_last=False,
    )

    # Load a fresh pretrained BERT for every learning rate.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
    ).to(DEVICE)

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = ClassWeightedFocalLoss(
        class_weights=class_weights,
        gamma=FOCAL_GAMMA,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )

    updates_per_epoch = math.ceil(
        len(train_loader) / GRADIENT_ACCUMULATION_STEPS
    )

    total_training_steps = updates_per_epoch * EPOCHS

    warmup_steps = int(
        total_training_steps * WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    best_macro_f1 = -1.0
    best_epoch = 0

    history = []

    training_start = time.perf_counter()

    print("\n" + "=" * 70)
    print(f"Experiment: {lr_tag}")
    print(f"Learning rate: {learning_rate}")
    print(f"Device: {DEVICE}")
    print(f"AMP enabled: {USE_AMP}")
    print(f"Train samples: {len(train_df):,}")
    print(f"Validation samples: {len(validation_df):,}")
    print(
        "Class weights:",
        np.round(class_weights_np, 4).tolist(),
    )
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):

        epoch_start = time.perf_counter()

        model.train()
        optimizer.zero_grad(set_to_none=True)

        train_loss_sum = 0.0
        train_samples = 0

        for step, batch in enumerate(train_loader):

            labels = batch.pop("labels").to(DEVICE)

            inputs = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):

                outputs = model(**inputs)

                loss = criterion(
                    outputs.logits,
                    labels,
                )

            # Account for the final incomplete accumulation group.
            group_start = (
                step // GRADIENT_ACCUMULATION_STEPS
            ) * GRADIENT_ACCUMULATION_STEPS

            group_size = min(
                GRADIENT_ACCUMULATION_STEPS,
                len(train_loader) - group_start,
            )

            scaled_loss = loss / group_size

            scaler.scale(scaled_loss).backward()

            batch_size = labels.size(0)

            train_loss_sum += (
                loss.detach().item() * batch_size
            )

            train_samples += batch_size

            is_update_step = (
                (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                or (step + 1) == len(train_loader)
            )

            if is_update_step:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        train_loss = (
            train_loss_sum / max(train_samples, 1)
        )

        val_metrics, y_true, y_pred = evaluate(
            model,
            validation_loader,
            criterion,
        )

        epoch_minutes = (
            time.perf_counter() - epoch_start
        ) / 60.0

        row = {
            "learning_rate": learning_rate,
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
            "epoch_minutes": epoch_minutes,
        }

        history.append(row)

        pd.DataFrame(history).to_csv(
            output_dir / "training_history.csv",
            index=False,
        )

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={val_metrics['validation_loss']:.5f} | "
            f"val_accuracy={val_metrics['accuracy']:.5f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.5f} | "
            f"time={epoch_minutes:.2f} min"
        )

        # Select checkpoint using Validation Macro F1 only.
        if val_metrics["macro_f1"] > best_macro_f1:

            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch

            checkpoint = {
                "model_name": MODEL_NAME,
                "num_classes": NUM_CLASSES,
                "max_length": MAX_LENGTH,
                "learning_rate": learning_rate,
                "focal_gamma": FOCAL_GAMMA,
                "epoch": epoch,
                "best_validation_macro_f1": best_macro_f1,
                "model_state_dict": model.state_dict(),
            }

            torch.save(
                checkpoint,
                output_dir / "best.pt",
            )

            save_validation_artifacts(
                output_dir,
                model,
                tokenizer,
                y_true,
                y_pred,
            )

            with open(
                output_dir / "best_metrics.json",
                "w",
                encoding="utf-8",
            ) as file:

                json.dump(
                    row,
                    file,
                    ensure_ascii=False,
                    indent=2,
                )

            print(
                f"  Saved best checkpoint: "
                f"epoch={best_epoch}, "
                f"Macro F1={best_macro_f1:.5f}"
            )

    training_minutes = (
        time.perf_counter() - training_start
    ) / 60.0

    result = {
        "learning_rate": learning_rate,
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_macro_f1,
        "training_minutes": training_minutes,
        "output_dir": str(output_dir),
    }

    with open(
        output_dir / "experiment_summary.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            result,
            file,
            ensure_ascii=False,
            indent=2,
        )

    del model
    del optimizer
    del scheduler
    del scaler
    del criterion
    del train_loader
    del validation_loader
    del train_dataset
    del validation_dataset

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 10. Main
# ============================================================

def main():

    set_seed(SEED)

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Device: {DEVICE}")

    if DEVICE.type == "cuda":
        print(
            "GPU:",
            torch.cuda.get_device_name(0),
        )

    print(f"AMP enabled: {USE_AMP}")

    print(
        "Only train.csv and validation.csv will be loaded. "
        "Test data will not be used."
    )

    # Load the existing fixed splits.
    train_df = load_split(
        TRAIN_PATH,
        "Train",
    )

    validation_df = load_split(
        VALIDATION_PATH,
        "Validation",
    )

    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(validation_df):,}")

    # Calculate weights from training labels only.
    present_classes = np.sort(
        train_df["label"].unique()
    )

    expected_classes = np.arange(NUM_CLASSES)

    if not np.array_equal(
        present_classes,
        expected_classes,
    ):

        raise ValueError(
            "Expected all labels 0..5 in the training split. "
            f"Found: {present_classes.tolist()}"
        )

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=expected_classes,
        y=train_df["label"].to_numpy(),
    ).astype(np.float32)

    print("\nTraining class weights:")

    for class_id, weight in enumerate(class_weights):
        print(
            f"Class {class_id}: {weight:.4f}"
        )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        raise ValueError(
            "Tokenizer has no pad token."
        )

    # Save experiment configuration.
    config = {
        "seed": SEED,
        "model_name": MODEL_NAME,
        "num_classes": NUM_CLASSES,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "effective_batch_size_approx": (
            BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
        ),
        "epochs": EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "focal_gamma": FOCAL_GAMMA,
        "learning_rates": LEARNING_RATES,
        "class_weights_from_train_only": (
            class_weights.tolist()
        ),
        "train_rows": len(train_df),
        "validation_rows": len(validation_df),
        "test_split_loaded": False,
    }

    with open(
        OUTPUT_ROOT / "experiment_config.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            config,
            file,
            ensure_ascii=False,
            indent=2,
        )

    results = []

    for learning_rate in LEARNING_RATES:

        result = run_experiment(
            learning_rate=learning_rate,
            train_df=train_df,
            validation_df=validation_df,
            tokenizer=tokenizer,
            class_weights_np=class_weights,
        )

        results.append(result)

    # Sort by the best Validation Macro F1.
    summary = pd.DataFrame(results).sort_values(
        "best_validation_macro_f1",
        ascending=False,
    )

    summary.to_csv(
        OUTPUT_ROOT / "learning_rate_comparison_summary.csv",
        index=False,
    )

    print("\n" + "=" * 70)
    print("LEARNING RATE COMPARISON SUMMARY")
    print("=" * 70)

    print(
        summary.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )

    print(
        "\nSummary saved to:",
        OUTPUT_ROOT / "learning_rate_comparison_summary.csv",
    )

    print(
        "\nTest data was not used. "
        "Select the learning rate using validation results."
    )


if __name__ == "__main__":
    main()

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
AMP enabled: True
Only train.csv and validation.csv will be loaded. Test data will not be used.
Train rows: 305,345
Validation rows: 65,422

Training class weights:
Class 0: 0.5733
Class 1: 0.4929
Class 2: 2.0086
Class 3: 1.2125
Class 4: 1.4523
Class 5: 4.6306


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: lr_1e-05
Learning rate: 1e-05
Device: cuda
AMP enabled: True
Train samples: 305,345
Validation samples: 65,422
Class weights: [0.5733000040054321, 0.492900013923645, 2.0085999965667725, 1.212499976158142, 1.452299952507019, 4.6305999755859375]
Epoch 1/3 | train_loss=0.21066 | val_loss=0.03120 | val_accuracy=0.94190 | val_macro_f1=0.91708 | time=9.21 min
  Saved best checkpoint: epoch=1, Macro F1=0.91708
Epoch 2/3 | train_loss=0.03064 | val_loss=0.02954 | val_accuracy=0.94240 | val_macro_f1=0.91759 | time=9.25 min
  Saved best checkpoint: epoch=2, Macro F1=0.91759
Epoch 3/3 | train_loss=0.02735 | val_loss=0.02957 | val_accuracy=0.94292 | val_macro_f1=0.91814 | time=9.01 min
  Saved best checkpoint: epoch=3, Macro F1=0.91814


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: lr_2e-05
Learning rate: 2e-05
Device: cuda
AMP enabled: True
Train samples: 305,345
Validation samples: 65,422
Class weights: [0.5733000040054321, 0.492900013923645, 2.0085999965667725, 1.212499976158142, 1.452299952507019, 4.6305999755859375]
Epoch 1/3 | train_loss=0.16126 | val_loss=0.03120 | val_accuracy=0.94195 | val_macro_f1=0.91723 | time=8.95 min
  Saved best checkpoint: epoch=1, Macro F1=0.91723
Epoch 2/3 | train_loss=0.02952 | val_loss=0.02935 | val_accuracy=0.94274 | val_macro_f1=0.91790 | time=8.69 min
  Saved best checkpoint: epoch=2, Macro F1=0.91790
Epoch 3/3 | train_loss=0.02612 | val_loss=0.02994 | val_accuracy=0.94303 | val_macro_f1=0.91832 | time=8.59 min
  Saved best checkpoint: epoch=3, Macro F1=0.91832


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: lr_3e-05
Learning rate: 3e-05
Device: cuda
AMP enabled: True
Train samples: 305,345
Validation samples: 65,422
Class weights: [0.5733000040054321, 0.492900013923645, 2.0085999965667725, 1.212499976158142, 1.452299952507019, 4.6305999755859375]
Epoch 1/3 | train_loss=0.14115 | val_loss=0.03185 | val_accuracy=0.94179 | val_macro_f1=0.91687 | time=8.63 min
  Saved best checkpoint: epoch=1, Macro F1=0.91687
Epoch 2/3 | train_loss=0.02953 | val_loss=0.02927 | val_accuracy=0.94274 | val_macro_f1=0.91784 | time=8.63 min
  Saved best checkpoint: epoch=2, Macro F1=0.91784
Epoch 3/3 | train_loss=0.02612 | val_loss=0.03037 | val_accuracy=0.94288 | val_macro_f1=0.91805 | time=8.62 min
  Saved best checkpoint: epoch=3, Macro F1=0.91805

LEARNING RATE COMPARISON SUMMARY
 learning_rate  best_epoch  best_validation_macro_f1  training_minutes                                             output_dir
      0.000020           3                  0.918319         26.604439 results/optimization/le

In [1]:
# 07_3_batch_size_comparison.py

import gc
import json
import math
import random
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "google-bert/bert-base-uncased"

TRAIN_PATH = Path("results/splits/train.csv")
VALIDATION_PATH = Path("results/splits/validation.csv")

OUTPUT_DIR = Path(
    "results/optimization/batch_size_comparison"
)

NUM_CLASSES = 6
MAX_LENGTH = 128

# Physical batch size to compare
BATCH_SIZES = [8, 16, 32]

# Keep gradient accumulation fixed.
GRAD_ACCUM_STEPS = 4

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

FOCAL_GAMMA = 2.0

NUM_WORKERS = 0
TOKENIZE_CHUNK_SIZE = 4096

USE_AMP = torch.cuda.is_available()
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Improves reproducibility, but some CUDA operations may
    # still vary slightly depending on hardware and libraries.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. Focal Loss
# ============================================================

class WeightedFocalLoss(nn.Module):
    """
    Class-weighted Focal Loss.

    CE_i = -log(p_t)
    FL_i = alpha_t * (1 - p_t)^gamma * CE_i

    Batch loss is normalized by the sum of alpha_t values.
    """

    def __init__(self, class_weights, gamma=2.0):
        super().__init__()

        self.register_buffer(
            "class_weights",
            class_weights.float()
        )

        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        # p_t = exp(-CE)
        p_t = torch.exp(-ce_loss)

        focal_factor = (1.0 - p_t) ** self.gamma

        alpha_t = self.class_weights[targets]

        loss = alpha_t * focal_factor * ce_loss

        denominator = alpha_t.sum().clamp_min(1e-12)

        return loss.sum() / denominator


# ============================================================
# 4. Dataset
# ============================================================

class EncodedTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": self.encodings["attention_mask"][index],
            "labels": int(self.labels[index]),
        }


# ============================================================
# 5. Tokenization
# ============================================================

def tokenize_in_chunks(texts, tokenizer):
    """
    Tokenize in chunks to avoid passing the entire dataset
    to the tokenizer in one large operation.

    Padding is performed dynamically inside each batch.
    """

    all_input_ids = []
    all_attention_masks = []

    total = len(texts)

    for start in range(0, total, TOKENIZE_CHUNK_SIZE):
        end = min(start + TOKENIZE_CHUNK_SIZE, total)

        chunk = texts[start:end]

        encoded = tokenizer(
            chunk,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_attention_mask=True,
        )

        all_input_ids.extend(encoded["input_ids"])
        all_attention_masks.extend(encoded["attention_mask"])

        print(
            f"\rTokenizing: {end:,}/{total:,}",
            end="",
            flush=True
        )

    print()

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
    }


# ============================================================
# 6. Load fixed train/validation splits
# ============================================================

def load_data():
    if not TRAIN_PATH.exists():
        raise FileNotFoundError(
            f"Train file not found: {TRAIN_PATH}"
        )

    if not VALIDATION_PATH.exists():
        raise FileNotFoundError(
            f"Validation file not found: {VALIDATION_PATH}"
        )

    # Intentionally do not load test.csv.
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VALIDATION_PATH)

    required_columns = {"text", "label"}

    for split_name, df in [
        ("train", train_df),
        ("validation", val_df),
    ]:
        missing = required_columns - set(df.columns)

        if missing:
            raise ValueError(
                f"{split_name}.csv is missing columns: {missing}"
            )

        if df["label"].isna().any():
            raise ValueError(
                f"{split_name}.csv contains missing labels."
            )

        if df["text"].isna().any():
            raise ValueError(
                f"{split_name}.csv contains missing text."
            )

        df["label"] = df["label"].astype(int)

        if not df["label"].isin(range(NUM_CLASSES)).all():
            raise ValueError(
                f"{split_name}.csv contains labels outside "
                f"0 to {NUM_CLASSES - 1}."
            )

    train_texts = (
        train_df["text"].astype(str).tolist()
    )

    val_texts = (
        val_df["text"].astype(str).tolist()
    )

    y_train = train_df["label"].to_numpy(dtype=np.int64)
    y_val = val_df["label"].to_numpy(dtype=np.int64)

    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(val_df):,}")
    print("Test data will not be loaded or evaluated.")

    return train_texts, y_train, val_texts, y_val


# ============================================================
# 7. Metrics
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }


# ============================================================
# 8. Validation
# ============================================================

def evaluate(
    model,
    data_loader,
    criterion,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    batch["labels"],
                )

            batch_size = batch["labels"].size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            predictions = torch.argmax(
                logits,
                dim=1,
            )

            all_labels.extend(
                batch["labels"].cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

    avg_loss = total_loss / max(total_samples, 1)

    metrics = calculate_metrics(
        all_labels,
        all_predictions,
    )

    metrics["loss"] = avg_loss

    return (
        metrics,
        np.asarray(all_labels),
        np.asarray(all_predictions),
    )


# ============================================================
# 9. Save validation artifacts
# ============================================================

def save_validation_artifacts(
    output_dir,
    model,
    tokenizer,
    y_true,
    y_pred,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(report).transpose().to_csv(
        output_dir / "validation_classification_report.csv",
        encoding="utf-8-sig",
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
    )

    pd.DataFrame(
        cm,
        index=[f"true_{i}" for i in range(NUM_CLASSES)],
        columns=[f"pred_{i}" for i in range(NUM_CLASSES)],
    ).to_csv(
        output_dir / "validation_confusion_matrix.csv",
        encoding="utf-8-sig",
    )

    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(cm, interpolation="nearest")

    ax.set_title("Validation Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(image, ax=ax)
    fig.tight_layout()

    fig.savefig(
        output_dir / "validation_confusion_matrix.png",
        dpi=200,
    )

    plt.close(fig)

    # Save best model and tokenizer.
    model.save_pretrained(
        output_dir / "best_model"
    )

    tokenizer.save_pretrained(
        output_dir / "best_model"
    )


# ============================================================
# 10. Training one batch-size experiment
# ============================================================

def run_experiment(
    batch_size,
    train_dataset,
    val_dataset,
    tokenizer,
    y_train,
    class_weights,
):
    set_seed(SEED)

    experiment_name = f"batch_size_{batch_size}"

    output_dir = OUTPUT_DIR / experiment_name
    output_dir.mkdir(parents=True, exist_ok=True)

    effective_batch_size = (
        batch_size * GRAD_ACCUM_STEPS
    )

    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=DataCollatorWithPadding(
            tokenizer=tokenizer,
            return_tensors="pt",
        ),
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=DataCollatorWithPadding(
            tokenizer=tokenizer,
            return_tensors="pt",
        ),
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
    ).to(DEVICE)

    criterion = WeightedFocalLoss(
        class_weights=class_weights.to(DEVICE),
        gamma=FOCAL_GAMMA,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    optimizer_steps_per_epoch = math.ceil(
        len(train_loader) / GRAD_ACCUM_STEPS
    )

    total_optimizer_steps = (
        optimizer_steps_per_epoch * EPOCHS
    )

    warmup_steps = int(
        total_optimizer_steps * WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    try:
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(
            enabled=USE_AMP,
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    history = []

    best_macro_f1 = -1.0
    best_epoch = None
    best_y_true = None
    best_y_pred = None
    best_metrics = None

    experiment_start = time.perf_counter()

    print("\n" + "=" * 72)
    print(f"Experiment: {experiment_name}")
    print(f"Physical Batch Size: {batch_size}")
    print(f"Gradient Accumulation: {GRAD_ACCUM_STEPS}")
    print(f"Effective Batch Size: {effective_batch_size}")
    print(f"Learning Rate: {LEARNING_RATE}")
    print(f"Device: {DEVICE}")
    print(f"AMP enabled: {USE_AMP}")
    print("=" * 72)

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()

        model.train()

        total_train_loss = 0.0
        total_train_samples = 0

        optimizer.zero_grad(set_to_none=True)

        total_batches = len(train_loader)

        for step, batch in enumerate(train_loader):
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            labels = batch["labels"]
            current_batch_size = labels.size(0)

            # Determine the number of microbatches in the
            # current accumulation group. The final group
            # may contain fewer than GRAD_ACCUM_STEPS batches.
            group_start = (
                step // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_size = min(
                GRAD_ACCUM_STEPS,
                total_batches - group_start,
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels,
                )

                scaled_loss = loss / group_size

            scaler.scale(scaled_loss).backward()

            total_train_loss += (
                loss.detach().item() * current_batch_size
            )

            total_train_samples += current_batch_size

            is_accumulation_boundary = (
                (step + 1) % GRAD_ACCUM_STEPS == 0
            )

            is_last_batch = (
                step + 1 == total_batches
            )

            if is_accumulation_boundary or is_last_batch:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        train_loss = (
            total_train_loss /
            max(total_train_samples, 1)
        )

        val_metrics, y_true, y_pred = evaluate(
            model,
            val_loader,
            criterion,
        )

        epoch_seconds = (
            time.perf_counter() - epoch_start
        )

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "learning_rate": scheduler.get_last_lr()[0],
            "epoch_minutes": epoch_seconds / 60.0,
        }

        history.append(record)

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={val_metrics['loss']:.5f} | "
            f"val_accuracy={val_metrics['accuracy']:.5f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.5f} | "
            f"time={epoch_seconds / 60.0:.2f} min"
        )

        # Select the checkpoint using Validation Macro F1.
        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch

            best_metrics = val_metrics.copy()
            best_y_true = y_true.copy()
            best_y_pred = y_pred.copy()

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_validation_macro_f1": best_macro_f1,
                    "batch_size": batch_size,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "effective_batch_size": effective_batch_size,
                    "learning_rate": LEARNING_RATE,
                    "seed": SEED,
                },
                output_dir / "best.pt",
            )

            print(
                f"  Saved best checkpoint: "
                f"epoch={epoch}, "
                f"Macro F1={best_macro_f1:.5f}"
            )

        pd.DataFrame(history).to_csv(
            output_dir / "training_history.csv",
            index=False,
            encoding="utf-8-sig",
        )

    # Load the best checkpoint before saving final artifacts.
    checkpoint = torch.load(
        output_dir / "best.pt",
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    save_validation_artifacts(
        output_dir=output_dir,
        model=model,
        tokenizer=tokenizer,
        y_true=best_y_true,
        y_pred=best_y_pred,
    )

    # Training curves
    history_df = pd.DataFrame(history)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(
        history_df["epoch"],
        history_df["train_loss"],
        marker="o",
        label="Train Loss",
    )

    axes[0].plot(
        history_df["epoch"],
        history_df["val_loss"],
        marker="o",
        label="Validation Loss",
    )

    axes[0].set_title("Loss by Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_xticks(history_df["epoch"])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        history_df["epoch"],
        history_df["val_macro_f1"],
        marker="o",
        label="Validation Macro F1",
    )

    axes[1].set_title("Validation Macro F1 by Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].set_xticks(history_df["epoch"])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir / "training_curves.png",
        dpi=200,
    )

    plt.close(fig)

    total_minutes = (
        time.perf_counter() - experiment_start
    ) / 60.0

    peak_gpu_memory_mb = None

    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = (
            torch.cuda.max_memory_allocated(DEVICE)
            / (1024 ** 2)
        )

    result = {
        "status": "success",
        "batch_size": batch_size,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": effective_batch_size,
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_metrics["macro_f1"],
        "best_validation_accuracy": best_metrics["accuracy"],
        "best_validation_loss": best_metrics["loss"],
        "best_validation_macro_precision": best_metrics["macro_precision"],
        "best_validation_macro_recall": best_metrics["macro_recall"],
        "best_validation_weighted_f1": best_metrics["weighted_f1"],
        "training_minutes": total_minutes,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "output_dir": str(output_dir),
    }

    with open(
        output_dir / "experiment_config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                **result,
                "model_name": MODEL_NAME,
                "learning_rate": LEARNING_RATE,
                "max_length": MAX_LENGTH,
                "epochs": EPOCHS,
                "weight_decay": WEIGHT_DECAY,
                "warmup_ratio": WARMUP_RATIO,
                "focal_gamma": FOCAL_GAMMA,
                "seed": SEED,
                "use_amp": USE_AMP,
                "device": str(DEVICE),
            },
            f,
            ensure_ascii=False,
            indent=4,
        )

    del model, optimizer, scheduler, scaler
    del train_loader, val_loader, criterion

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 11. Main
# ============================================================

def main():
    set_seed(SEED)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Device: {DEVICE}")

    if DEVICE.type == "cuda":
        print(
            "GPU:",
            torch.cuda.get_device_name(DEVICE),
        )

    print("AMP enabled:", USE_AMP)
    print("Test data will not be used.")

    # Load only fixed train and validation splits.
    (
        train_texts,
        y_train,
        val_texts,
        y_val,
    ) = load_data()

    # Calculate class weights from training labels only.
    classes = np.arange(NUM_CLASSES)

    calculated_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train,
    )

    class_weights = torch.tensor(
        calculated_weights,
        dtype=torch.float32,
    )

    print("\nTraining class weights:")

    for class_id, weight in enumerate(calculated_weights):
        print(
            f"Class {class_id}: {weight:.4f}"
        )

    # Load tokenizer and tokenize once for all experiments.
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    print("\nTokenizing training data...")
    train_encodings = tokenize_in_chunks(
        train_texts,
        tokenizer,
    )

    print("\nTokenizing validation data...")
    val_encodings = tokenize_in_chunks(
        val_texts,
        tokenizer,
    )

    train_dataset = EncodedTextDataset(
        train_encodings,
        y_train,
    )

    val_dataset = EncodedTextDataset(
        val_encodings,
        y_val,
    )

    results = []

    for batch_size in BATCH_SIZES:
        try:
            result = run_experiment(
                batch_size=batch_size,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                tokenizer=tokenizer,
                y_train=y_train,
                class_weights=class_weights,
            )

            results.append(result)

        except RuntimeError as exc:
            # Continue only for CUDA out-of-memory errors.
            if (
                "out of memory" in str(exc).lower()
                and torch.cuda.is_available()
            ):
                print(
                    f"\nCUDA OOM at batch size {batch_size}. "
                    "This experiment is marked as failed."
                )

                results.append({
                    "status": "failed_oom",
                    "batch_size": batch_size,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "effective_batch_size": (
                        batch_size * GRAD_ACCUM_STEPS
                    ),
                    "output_dir": str(
                        OUTPUT_DIR / f"batch_size_{batch_size}"
                    ),
                    "error": str(exc),
                })

                gc.collect()
                torch.cuda.empty_cache()

            else:
                raise

    summary = pd.DataFrame(results)

    summary_path = (
        OUTPUT_DIR / "batch_size_comparison_summary.csv"
    )

    summary.to_csv(
        summary_path,
        index=False,
        encoding="utf-8-sig",
    )

    successful = summary[
        summary["status"] == "success"
    ].copy()

    if not successful.empty:
        successful = successful.sort_values(
            "best_validation_macro_f1",
            ascending=False,
        )

        print("\n" + "=" * 88)
        print("BATCH SIZE COMPARISON SUMMARY")
        print("=" * 88)

        display_columns = [
            "batch_size",
            "gradient_accumulation_steps",
            "effective_batch_size",
            "best_epoch",
            "best_validation_macro_f1",
            "best_validation_accuracy",
            "training_minutes",
            "peak_gpu_memory_mb",
        ]

        print(
            successful[display_columns].to_string(
                index=False
            )
        )

        # Comparison plot
        fig, ax = plt.subplots(figsize=(8, 5))

        ax.plot(
            successful["effective_batch_size"],
            successful["best_validation_macro_f1"],
            marker="o",
        )

        ax.set_title(
            "Effective Batch Size vs Validation Macro F1"
        )
        ax.set_xlabel("Effective Batch Size")
        ax.set_ylabel("Best Validation Macro F1")
        ax.grid(True, alpha=0.3)

        fig.tight_layout()

        fig.savefig(
            OUTPUT_DIR / "batch_size_comparison.png",
            dpi=200,
        )

        plt.close(fig)

        best_result = successful.iloc[0]

        print(
            "\nBest configuration by Validation Macro F1:"
        )
        print(
            f"Physical Batch Size: "
            f"{int(best_result['batch_size'])}"
        )
        print(
            f"Effective Batch Size: "
            f"{int(best_result['effective_batch_size'])}"
        )
        print(
            f"Best Validation Macro F1: "
            f"{best_result['best_validation_macro_f1']:.5f}"
        )

    else:
        print(
            "\nNo experiment completed successfully. "
            "Check the errors above."
        )

    print(f"\nSummary saved to: {summary_path}")
    print("Test data was not used.")


if __name__ == "__main__":
    main()

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
AMP enabled: True
Test data will not be used.
Train rows: 305,345
Validation rows: 65,422
Test data will not be loaded or evaluated.

Training class weights:
Class 0: 0.5733
Class 1: 0.4929
Class 2: 2.0086
Class 3: 1.2125
Class 4: 1.4523
Class 5: 4.6306

Tokenizing training data...
Tokenizing: 305,345/305,345

Tokenizing validation data...
Tokenizing: 65,422/65,422


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: batch_size_8
Physical Batch Size: 8
Gradient Accumulation: 4
Effective Batch Size: 32
Learning Rate: 2e-05
Device: cuda
AMP enabled: True
Epoch 1/3 | train_loss=0.13751 | val_loss=0.03161 | val_accuracy=0.94204 | val_macro_f1=0.91675 | time=13.32 min
  Saved best checkpoint: epoch=1, Macro F1=0.91675
Epoch 2/3 | train_loss=0.02990 | val_loss=0.03043 | val_accuracy=0.94283 | val_macro_f1=0.91809 | time=15.44 min
  Saved best checkpoint: epoch=2, Macro F1=0.91809
Epoch 3/3 | train_loss=0.02588 | val_loss=0.03118 | val_accuracy=0.94277 | val_macro_f1=0.91806 | time=15.72 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Experiment: batch_size_32
Physical Batch Size: 32
Gradient Accumulation: 4
Effective Batch Size: 128
Learning Rate: 2e-05
Device: cuda
AMP enabled: True
Epoch 1/3 | train_loss=0.19370 | val_loss=0.03001 | val_accuracy=0.94193 | val_macro_f1=0.91704 | time=5.76 min
  Saved best checkpoint: epoch=1, Macro F1=0.91704
Epoch 2/3 | train_loss=0.02959 | val_loss=0.02991 | val_accuracy=0.94214 | val_macro_f1=0.91730 | time=5.82 min
  Saved best checkpoint: epoch=2, Macro F1=0.91730
Epoch 3/3 | train_loss=0.02668 | val_loss=0.02929 | val_accuracy=0.94273 | val_macro_f1=0.91799 | time=5.94 min
  Saved best checkpoint: epoch=3, Macro F1=0.91799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BATCH SIZE COMPARISON SUMMARY
 batch_size  gradient_accumulation_steps  effective_batch_size  best_epoch  best_validation_macro_f1  best_validation_accuracy  training_minutes  peak_gpu_memory_mb
         16                            4                    64           3                  0.918319                  0.943031         27.449116         2672.657227
          8                            4                    32           2                  0.918093                  0.942833         45.205006         2531.048828
         32                            4                   128           3                  0.917987                  0.942726         18.419759         3427.588379

Best configuration by Validation Macro F1:
Physical Batch Size: 16
Effective Batch Size: 64
Best Validation Macro F1: 0.91832

Summary saved to: results/optimization/batch_size_comparison/batch_size_comparison_summary.csv
Test data was not used.


In [2]:
# 07_4_max_sequence_length_comparison.py

import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "google-bert/bert-base-uncased"

TRAIN_PATH = Path("results/splits/train.csv")
VALIDATION_PATH = Path("results/splits/validation.csv")

OUTPUT_DIR = Path(
    "results/optimization/max_sequence_length_comparison"
)

NUM_CLASSES = 6

# Experiment variable
MAX_LENGTHS = [64, 128, 256]

# Fixed training configuration
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
EPOCHS = 3

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
FOCAL_GAMMA = 2.0

NUM_WORKERS = 0
TOKENIZE_CHUNK_SIZE = 4096

USE_AMP = torch.cuda.is_available()
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. Weighted Focal Loss
# ============================================================

class WeightedFocalLoss(nn.Module):
    """
    CE_i = -log(p_t)

    Focal_i = alpha_t * (1 - p_t)^gamma * CE_i

    The batch loss is normalized by the sum of alpha_t.
    """

    def __init__(self, class_weights, gamma=2.0):
        super().__init__()

        self.register_buffer(
            "class_weights",
            class_weights.float(),
        )

        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none",
        )

        p_t = torch.exp(-ce_loss)

        focal_factor = (
            1.0 - p_t
        ).pow(self.gamma)

        alpha_t = self.class_weights[targets]

        loss = (
            alpha_t
            * focal_factor
            * ce_loss
        )

        denominator = alpha_t.sum().clamp_min(1e-12)

        return loss.sum() / denominator


# ============================================================
# 4. Dataset
# ============================================================

class EncodedTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = np.asarray(
            labels,
            dtype=np.int64,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": (
                self.encodings["attention_mask"][index]
            ),
            "labels": int(self.labels[index]),
        }


# ============================================================
# 5. Load fixed data
# ============================================================

def load_data():
    if not TRAIN_PATH.exists():
        raise FileNotFoundError(
            f"Training file not found: {TRAIN_PATH}"
        )

    if not VALIDATION_PATH.exists():
        raise FileNotFoundError(
            f"Validation file not found: {VALIDATION_PATH}"
        )

    # Deliberately do not load test.csv.
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VALIDATION_PATH)

    required_columns = {"text", "label"}

    for split_name, df in [
        ("train", train_df),
        ("validation", val_df),
    ]:
        missing = required_columns - set(df.columns)

        if missing:
            raise ValueError(
                f"{split_name}.csv missing columns: {missing}"
            )

        if df["text"].isna().any():
            raise ValueError(
                f"{split_name}.csv contains missing text."
            )

        if df["label"].isna().any():
            raise ValueError(
                f"{split_name}.csv contains missing labels."
            )

        df["label"] = df["label"].astype(int)

        if not df["label"].isin(
            range(NUM_CLASSES)
        ).all():
            raise ValueError(
                f"{split_name}.csv has labels outside "
                f"0 to {NUM_CLASSES - 1}."
            )

    train_texts = train_df["text"].astype(str).tolist()
    val_texts = val_df["text"].astype(str).tolist()

    y_train = train_df["label"].to_numpy(dtype=np.int64)
    y_val = val_df["label"].to_numpy(dtype=np.int64)

    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(val_df):,}")
    print("Test data will not be loaded or evaluated.")

    return train_texts, y_train, val_texts, y_val


# ============================================================
# 6. Tokenization for a specific maximum length
# ============================================================

def tokenize_in_chunks(texts, tokenizer, max_length):
    all_input_ids = []
    all_attention_masks = []

    total = len(texts)

    for start in range(0, total, TOKENIZE_CHUNK_SIZE):
        end = min(
            start + TOKENIZE_CHUNK_SIZE,
            total,
        )

        encoded = tokenizer(
            texts[start:end],
            truncation=True,
            max_length=max_length,
            padding=False,
            return_attention_mask=True,
        )

        all_input_ids.extend(encoded["input_ids"])
        all_attention_masks.extend(
            encoded["attention_mask"]
        )

        print(
            f"\rMax length {max_length}: "
            f"{end:,}/{total:,}",
            end="",
            flush=True,
        )

    print()

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
    }


def get_length_statistics(encodings):
    lengths = np.asarray(
        [
            len(ids)
            for ids in encodings["input_ids"]
        ],
        dtype=np.int32,
    )

    return {
        "mean_token_length": float(lengths.mean()),
        "median_token_length": float(
            np.median(lengths)
        ),
        "percent_at_max_length": float(
            (lengths >= CURRENT_MAX_LENGTH[0]).mean()
            * 100.0
        ),
    }


# ============================================================
# 7. Metrics
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }


# ============================================================
# 8. Validation
# ============================================================

def evaluate(model, data_loader, criterion):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    batch["labels"],
                )

            batch_n = batch["labels"].size(0)

            total_loss += loss.item() * batch_n
            total_samples += batch_n

            predictions = logits.argmax(dim=1)

            all_labels.extend(
                batch["labels"].cpu().tolist()
            )

            all_predictions.extend(
                predictions.cpu().tolist()
            )

    metrics = calculate_metrics(
        all_labels,
        all_predictions,
    )

    metrics["loss"] = (
        total_loss / max(total_samples, 1)
    )

    return (
        metrics,
        np.asarray(all_labels),
        np.asarray(all_predictions),
    )


# ============================================================
# 9. Save validation artifacts
# ============================================================

def save_validation_artifacts(
    output_dir,
    model,
    tokenizer,
    y_true,
    y_pred,
):
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(report).transpose().to_csv(
        output_dir / "validation_classification_report.csv",
        encoding="utf-8-sig",
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
    )

    pd.DataFrame(
        cm,
        index=[f"true_{i}" for i in range(NUM_CLASSES)],
        columns=[f"pred_{i}" for i in range(NUM_CLASSES)],
    ).to_csv(
        output_dir / "validation_confusion_matrix.csv",
        encoding="utf-8-sig",
    )

    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(cm, interpolation="nearest")

    ax.set_title("Validation Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(image, ax=ax)
    fig.tight_layout()

    fig.savefig(
        output_dir / "validation_confusion_matrix.png",
        dpi=200,
    )

    plt.close(fig)

    model.save_pretrained(
        output_dir / "best_model"
    )

    tokenizer.save_pretrained(
        output_dir / "best_model"
    )


# ============================================================
# 10. Train one maximum-length experiment
# ============================================================

def run_experiment(
    max_length,
    train_texts,
    y_train,
    val_texts,
    y_val,
    tokenizer,
    class_weights,
):
    set_seed(SEED)

    output_dir = OUTPUT_DIR / f"max_length_{max_length}"
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("\n" + "=" * 76)
    print(f"Experiment: max_length_{max_length}")
    print(f"Max Sequence Length: {max_length}")
    print(f"Physical Batch Size: {BATCH_SIZE}")
    print(f"Gradient Accumulation: {GRAD_ACCUM_STEPS}")
    print(
        "Effective Batch Size:",
        BATCH_SIZE * GRAD_ACCUM_STEPS,
    )
    print(f"Learning Rate: {LEARNING_RATE}")
    print(f"Device: {DEVICE}")
    print("=" * 76)

    train_encodings = tokenize_in_chunks(
        train_texts,
        tokenizer,
        max_length,
    )

    val_encodings = tokenize_in_chunks(
        val_texts,
        tokenizer,
        max_length,
    )

    train_lengths = np.asarray(
        [
            len(ids)
            for ids in train_encodings["input_ids"]
        ],
        dtype=np.int32,
    )

    val_lengths = np.asarray(
        [
            len(ids)
            for ids in val_encodings["input_ids"]
        ],
        dtype=np.int32,
    )

    length_stats = {
        "train_mean_token_length": float(
            train_lengths.mean()
        ),
        "train_median_token_length": float(
            np.median(train_lengths)
        ),
        "train_percent_at_max_length": float(
            (train_lengths >= max_length).mean() * 100
        ),
        "validation_mean_token_length": float(
            val_lengths.mean()
        ),
        "validation_median_token_length": float(
            np.median(val_lengths)
        ),
        "validation_percent_at_max_length": float(
            (val_lengths >= max_length).mean() * 100
        ),
    }

    print(
        "Mean token length (train): "
        f"{length_stats['train_mean_token_length']:.2f}"
    )

    print(
        "Train sequences reaching configured max length: "
        f"{length_stats['train_percent_at_max_length']:.2f}%"
    )

    train_dataset = EncodedTextDataset(
        train_encodings,
        y_train,
    )

    val_dataset = EncodedTextDataset(
        val_encodings,
        y_val,
    )

    del train_encodings, val_encodings
    gc.collect()

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt",
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
    ).to(DEVICE)

    criterion = WeightedFocalLoss(
        class_weights=class_weights.to(DEVICE),
        gamma=FOCAL_GAMMA,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    optimizer_steps_per_epoch = math.ceil(
        len(train_loader) / GRAD_ACCUM_STEPS
    )

    total_optimizer_steps = (
        optimizer_steps_per_epoch * EPOCHS
    )

    warmup_steps = int(
        total_optimizer_steps * WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    try:
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(
            enabled=USE_AMP,
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    history = []

    best_macro_f1 = -1.0
    best_epoch = None
    best_metrics = None
    best_y_true = None
    best_y_pred = None

    experiment_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()

        model.train()

        total_train_loss = 0.0
        total_train_samples = 0

        optimizer.zero_grad(set_to_none=True)

        # Group microbatches so the final incomplete
        # accumulation group is handled correctly.
        iterator = iter(train_loader)
        num_batches = len(train_loader)
        batches_processed = 0

        while batches_processed < num_batches:
            group = []

            for _ in range(GRAD_ACCUM_STEPS):
                if batches_processed >= num_batches:
                    break

                try:
                    batch = next(iterator)
                except StopIteration:
                    break

                group.append(batch)
                batches_processed += 1

            if not group:
                break

            group_sample_count = sum(
                batch["labels"].size(0)
                for batch in group
            )

            for batch in group:
                batch = {
                    key: value.to(DEVICE)
                    for key, value in batch.items()
                }

                labels = batch["labels"]
                batch_n = labels.size(0)

                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=USE_AMP,
                ):
                    outputs = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                    )

                    logits = outputs.logits

                    loss = criterion(
                        logits,
                        labels,
                    )

                    # Weight each microbatch by its share
                    # of examples in the accumulation group.
                    scaled_loss = (
                        loss
                        * batch_n
                        / group_sample_count
                    )

                scaler.scale(scaled_loss).backward()

                total_train_loss += (
                    loss.detach().item() * batch_n
                )

                total_train_samples += batch_n

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        train_loss = (
            total_train_loss /
            max(total_train_samples, 1)
        )

        val_metrics, y_true, y_pred = evaluate(
            model,
            val_loader,
            criterion,
        )

        epoch_seconds = (
            time.perf_counter() - epoch_start
        )

        record = {
            "max_length": max_length,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "epoch_minutes": epoch_seconds / 60.0,
        }

        history.append(record)

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={val_metrics['loss']:.5f} | "
            f"val_accuracy={val_metrics['accuracy']:.5f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.5f} | "
            f"time={epoch_seconds / 60.0:.2f} min"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch

            best_metrics = val_metrics.copy()
            best_y_true = y_true.copy()
            best_y_pred = y_pred.copy()

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "best_validation_macro_f1": best_macro_f1,
                    "max_length": max_length,
                    "batch_size": BATCH_SIZE,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "learning_rate": LEARNING_RATE,
                    "seed": SEED,
                },
                output_dir / "best.pt",
            )

            print(
                f"  Saved best checkpoint: "
                f"epoch={epoch}, "
                f"Macro F1={best_macro_f1:.5f}"
            )

        pd.DataFrame(history).to_csv(
            output_dir / "training_history.csv",
            index=False,
            encoding="utf-8-sig",
        )

    # Restore best checkpoint.
    checkpoint = torch.load(
        output_dir / "best.pt",
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    save_validation_artifacts(
        output_dir=output_dir,
        model=model,
        tokenizer=tokenizer,
        y_true=best_y_true,
        y_pred=best_y_pred,
    )

    # Training curves
    history_df = pd.DataFrame(history)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
    )

    axes[0].plot(
        history_df["epoch"],
        history_df["train_loss"],
        marker="o",
        label="Train Loss",
    )

    axes[0].plot(
        history_df["epoch"],
        history_df["val_loss"],
        marker="o",
        label="Validation Loss",
    )

    axes[0].set_title("Loss by Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_xticks(history_df["epoch"])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        history_df["epoch"],
        history_df["val_macro_f1"],
        marker="o",
    )

    axes[1].set_title("Validation Macro F1 by Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].set_xticks(history_df["epoch"])
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir / "training_curves.png",
        dpi=200,
    )

    plt.close(fig)

    total_minutes = (
        time.perf_counter() - experiment_start
    ) / 60.0

    peak_gpu_memory_mb = None

    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = (
            torch.cuda.max_memory_allocated(DEVICE)
            / (1024 ** 2)
        )

    result = {
        "status": "success",
        "max_length": max_length,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": (
            BATCH_SIZE * GRAD_ACCUM_STEPS
        ),
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_metrics["macro_f1"],
        "best_validation_accuracy": best_metrics["accuracy"],
        "best_validation_loss": best_metrics["loss"],
        "best_validation_macro_precision": best_metrics["macro_precision"],
        "best_validation_macro_recall": best_metrics["macro_recall"],
        "best_validation_weighted_f1": best_metrics["weighted_f1"],
        "training_minutes": total_minutes,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        **length_stats,
        "output_dir": str(output_dir),
    }

    with open(
        output_dir / "experiment_config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                **result,
                "model_name": MODEL_NAME,
                "learning_rate": LEARNING_RATE,
                "epochs": EPOCHS,
                "weight_decay": WEIGHT_DECAY,
                "warmup_ratio": WARMUP_RATIO,
                "focal_gamma": FOCAL_GAMMA,
                "seed": SEED,
                "use_amp": USE_AMP,
                "device": str(DEVICE),
            },
            f,
            ensure_ascii=False,
            indent=4,
        )

    del model, optimizer, scheduler, scaler
    del train_loader, val_loader
    del train_dataset, val_dataset, criterion

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 11. Main
# ============================================================

def main():
    set_seed(SEED)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Device: {DEVICE}")

    if DEVICE.type == "cuda":
        print(
            "GPU:",
            torch.cuda.get_device_name(DEVICE),
        )

    print(f"AMP enabled: {USE_AMP}")
    print("Test data will not be used.")

    (
        train_texts,
        y_train,
        val_texts,
        y_val,
    ) = load_data()

    # Compute class weights using training labels only.
    class_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y_train,
    )

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32,
    )

    print("\nTraining class weights:")

    for class_id, weight in enumerate(class_weights_np):
        print(
            f"Class {class_id}: {weight:.4f}"
        )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    results = []

    for max_length in MAX_LENGTHS:
        try:
            result = run_experiment(
                max_length=max_length,
                train_texts=train_texts,
                y_train=y_train,
                val_texts=val_texts,
                y_val=y_val,
                tokenizer=tokenizer,
                class_weights=class_weights,
            )

            results.append(result)

        except RuntimeError as exc:
            if (
                "out of memory" in str(exc).lower()
                and torch.cuda.is_available()
            ):
                print(
                    f"\nCUDA OOM at max_length={max_length}. "
                    "Marking this experiment as failed."
                )

                results.append({
                    "status": "failed_oom",
                    "max_length": max_length,
                    "batch_size": BATCH_SIZE,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "effective_batch_size": (
                        BATCH_SIZE * GRAD_ACCUM_STEPS
                    ),
                    "error": str(exc),
                    "output_dir": str(
                        OUTPUT_DIR / f"max_length_{max_length}"
                    ),
                })

                gc.collect()
                torch.cuda.empty_cache()

            else:
                raise

    summary = pd.DataFrame(results)

    summary_path = (
        OUTPUT_DIR / "max_sequence_length_comparison_summary.csv"
    )

    summary.to_csv(
        summary_path,
        index=False,
        encoding="utf-8-sig",
    )

    successful = summary[
        summary["status"] == "success"
    ].copy()

    if not successful.empty:
        successful = successful.sort_values(
            "best_validation_macro_f1",
            ascending=False,
        )

        display_columns = [
            "max_length",
            "best_epoch",
            "best_validation_macro_f1",
            "best_validation_accuracy",
            "training_minutes",
            "peak_gpu_memory_mb",
            "train_mean_token_length",
            "train_percent_at_max_length",
        ]

        print("\n" + "=" * 100)
        print("MAX SEQUENCE LENGTH COMPARISON SUMMARY")
        print("=" * 100)

        print(
            successful[display_columns].to_string(
                index=False
            )
        )

        # Plot Validation Macro F1 vs maximum length.
        fig, ax = plt.subplots(figsize=(8, 5))

        ax.plot(
            successful["max_length"],
            successful["best_validation_macro_f1"],
            marker="o",
        )

        ax.set_title(
            "Max Sequence Length vs Validation Macro F1"
        )
        ax.set_xlabel("Max Sequence Length")
        ax.set_ylabel("Best Validation Macro F1")
        ax.set_xticks(successful["max_length"])
        ax.grid(True, alpha=0.3)

        fig.tight_layout()

        fig.savefig(
            OUTPUT_DIR / "max_sequence_length_comparison.png",
            dpi=200,
        )

        plt.close(fig)

        best_result = successful.iloc[0]

        print("\nBest configuration by Validation Macro F1:")
        print(
            f"Max Sequence Length: "
            f"{int(best_result['max_length'])}"
        )
        print(
            f"Validation Macro F1: "
            f"{best_result['best_validation_macro_f1']:.5f}"
        )

    else:
        print(
            "\nNo experiment completed successfully. "
            "Check the errors above."
        )

    print(f"\nSummary saved to: {summary_path}")
    print("Test data was not used.")


if __name__ == "__main__":
    main()

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
AMP enabled: True
Test data will not be used.
Train rows: 305,345
Validation rows: 65,422
Test data will not be loaded or evaluated.

Training class weights:
Class 0: 0.5733
Class 1: 0.4929
Class 2: 2.0086
Class 3: 1.2125
Class 4: 1.4523
Class 5: 4.6306

Experiment: max_length_64
Max Sequence Length: 64
Physical Batch Size: 16
Gradient Accumulation: 4
Effective Batch Size: 64
Learning Rate: 2e-05
Device: cuda
Max length 64: 305,345/305,345
Max length 64: 65,422/65,422
Mean token length (train): 22.31
Train sequences reaching configured max length: 0.25%


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train_loss=0.16299 | val_loss=0.03099 | val_accuracy=0.94213 | val_macro_f1=0.91729 | time=6.80 min
  Saved best checkpoint: epoch=1, Macro F1=0.91729
Epoch 2/3 | train_loss=0.02981 | val_loss=0.02984 | val_accuracy=0.94251 | val_macro_f1=0.91773 | time=6.76 min
  Saved best checkpoint: epoch=2, Macro F1=0.91773
Epoch 3/3 | train_loss=0.02629 | val_loss=0.03003 | val_accuracy=0.94273 | val_macro_f1=0.91802 | time=6.80 min
  Saved best checkpoint: epoch=3, Macro F1=0.91802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train_loss=0.16126 | val_loss=0.03101 | val_accuracy=0.94195 | val_macro_f1=0.91723 | time=6.77 min
  Saved best checkpoint: epoch=1, Macro F1=0.91723
Epoch 2/3 | train_loss=0.02952 | val_loss=0.02941 | val_accuracy=0.94274 | val_macro_f1=0.91790 | time=6.77 min
  Saved best checkpoint: epoch=2, Macro F1=0.91790
Epoch 3/3 | train_loss=0.02612 | val_loss=0.02989 | val_accuracy=0.94303 | val_macro_f1=0.91832 | time=6.76 min
  Saved best checkpoint: epoch=3, Macro F1=0.91832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MAX SEQUENCE LENGTH COMPARISON SUMMARY
 max_length  best_epoch  best_validation_macro_f1  best_validation_accuracy  training_minutes  peak_gpu_memory_mb  train_mean_token_length  train_percent_at_max_length
        128           3                  0.918319                  0.943031         20.803809         2686.266602                22.322750                     0.000000
        256           3                  0.918319                  0.943031         20.712279         2697.586914                22.322750                     0.000000
         64           3                  0.918015                  0.942726         20.799825         2336.931152                22.314523                     0.246934

Best configuration by Validation Macro F1:
Max Sequence Length: 128
Validation Macro F1: 0.91832

Summary saved to: results/optimization/max_sequence_length_comparison/max_sequence_length_comparison_summary.csv
Test data was not used.
